<a href="https://colab.research.google.com/github/ryanngholston05/document-qa/blob/main/Lab7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --no-deps bitsandbytes accelerate xformers peft trl \
    triton cut_cross_entropy unsloth_zoo

!pip install -q sentencepiece protobuf "datasets>=3.4.1" \
    huggingface_hub hf_transfer

!pip install --no-deps unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 122.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 630.8/630.8 kB 55.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 415.2/415.2 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 62.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.4.2 requires msgspec, which is not installed.
unsloth-zoo 2026.4.2 requires tyro, which is not installed.
unsloth-zoo 2026.4.2 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 4.0.0 which is incompatible.
unsloth-zoo 2026.4.2 requires torchao>=0.13.0, but you have torchao 0.10.0 which is incompatible.
unsloth-zoo 2026.4.2 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,

In [2]:
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel, is_bfloat16_supported
from trl import SFTTrainer, SFTConfig
from peft import LoraConfig, get_peft_model

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
max_seq_length = 256

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=max_seq_length,
    dtype=None,          # Auto: float16 for T4, bfloat16 for Ampere+
    load_in_4bit=True,
)

==((====))==  Unsloth 2026.4.1: Fast Llama patching. Transformers: 5.0.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-bnb-4bit as a legacy tokenizer.


In [4]:
def generate_response(mdl, tokenizer, prompt, max_new_tokens=150):
    mdl.eval()
    messages = [
        {"role": "system", "content": "You are a medical AI assistant.\nAnswer questions accurately and concisely."},
        {"role": "user", "content": prompt}
    ]

    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(input_text, return_tensors="pt").to(mdl.device)

    with torch.no_grad():
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            use_cache=True,
        )

    response = outputs[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(response, skip_special_tokens=True)

In [7]:
test_prompts = [
    "What are the possible causes of low PTH and high calcium levels?",
    "A patient presents with low ejection fraction. What type of cardiac dysfunction is this most commonly associated with?",
    "What does the combination of very high hematocrit and low EPO suggest?",
    "What condition is suggested by low cortisol, low sodium, and high potassium in an infant?",
]

FastLanguageModel.for_inference(model)

baseline_outputs = []

for prompt in test_prompts:
    response = generate_response(model, tokenizer, prompt)
    baseline_outputs.append((prompt, response))

for i, (prompt, response) in enumerate(baseline_outputs, 1):
    print(f"Prompt {i}: {prompt}")
    print(f"Response: {response}")
    print("-" * 80)

Prompt 1: What are the possible causes of low PTH and high calcium levels?
Response: Low parathyroid hormone (PTH) and high calcium levels can be caused by several conditions. Some possible causes include:

1. **Hypercalcemia of malignancy**: Certain types of cancer, such as squamous cell lung cancer, renal cell carcinoma, and lymphoma, can produce substances that stimulate the release of calcium into the blood.
2. **Vitamin D intoxication**: Excessive intake of vitamin D or its analogs can lead to increased absorption of calcium from the gut, resulting in high calcium levels and suppressed PTH secretion.
3. **Exogenous calcium intake**: Consuming large amounts of calcium supplements or calcium-rich foods can cause hypercalcemia.
4. **Granulomatous diseases**: Conditions like sarcoidosis, tuberculosis
--------------------------------------------------------------------------------
Prompt 2: A patient presents with low ejection fraction. What type of cardiac dysfunction is this most com

## Baseline Responses (Before Fine-Tuning)

## Prompt 1: What are the possible causes of low PTH and high calcium levels?
Response: Low parathyroid hormone (PTH) and high calcium levels can be caused by several conditions. Some possible causes include:

1. **Hypercalcemia of malignancy**: Certain types of cancer, such as squamous cell lung cancer, renal cell carcinoma, and lymphoma, can produce substances that stimulate the release of calcium into the blood.
2. **Vitamin D intoxication**: Excessive intake of vitamin D or its analogs can lead to increased absorption of calcium from the gut, resulting in high calcium levels and suppressed PTH secretion.
3. **Exogenous calcium intake**: Consuming large amounts of calcium supplements or calcium-rich foods can cause hypercalcemia.
4. **Granulomatous diseases**: Conditions like sarcoidosis, tuberculosis
--------------------------------------------------------------------------------
Prompt 2: A patient presents with low ejection fraction. What type of cardiac dysfunction is this most commonly associated with?
Response: A patient with low ejection fraction is most commonly associated with systolic heart failure.

Ejection fraction (EF) is the percentage of blood that is pumped out of the left ventricle with each heartbeat. A normal ejection fraction is typically considered to be 55% or higher.

A low ejection fraction, often defined as less than 40%, is indicative of systolic dysfunction, which means the heart's ability to contract and pump blood efficiently is impaired. This is a hallmark of heart failure with reduced ejection fraction (HFrEF), which is a common type of heart failure.
--------------------------------------------------------------------------------
Prompt 3: What does the combination of very high hematocrit and low EPO suggest?
Response: A combination of very high hematocrit and low erythropoietin (EPO) levels can suggest a polycythemia vera (PV) or a similar myeloproliferative disorder.

In polycythemia vera, the bone marrow produces excessive red blood cells, white blood cells, and platelets, leading to an increased hematocrit. However, the EPO level is usually low because the body is not producing the normal feedback mechanism to regulate red blood cell production.

In the absence of other causes such as dehydration, anemia, or high-altitude adaptation, a low EPO level in the context of high hematocrit can point towards a myeloproliferative disorder like polycythemia vera.
--------------------------------------------------------------------------------
Prompt 4: What condition is suggested by low cortisol, low sodium, and high potassium in an infant?
Response: Low cortisol, low sodium, and high potassium in an infant are indicative of Congenital Adrenal Hyperplasia (CAH), specifically the salt-wasting form. CAH is a genetic disorder affecting the adrenal glands, which are responsible for producing hormones, including cortisol and aldosterone. The salt-wasting form of CAH leads to decreased aldosterone production, resulting in low sodium and high potassium levels, as well as cortisol deficiency.


In [8]:
dataset = load_dataset(
    "medalpaca/medical_meadow_medical_flashcards", split="train"
)

dataset = dataset.shuffle(seed=42).select(range(1000))

README.md: 0.00B [00:00, ?B/s]

medical_meadow_wikidoc_medical_flashcard(…):   0%|          | 0.00/17.7M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/33955 [00:00<?, ? examples/s]

In [9]:
def format_flashcard(example):
    text = (
        "<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n"
        "You are a medical AI assistant. Answer questions accurately "
        "and concisely.<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n\n"
        f"{example['input']}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n\n"
        f"{example['output']}<|eot_id|>"
    )
    return {"text": text}

dataset = dataset.map(format_flashcard)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [10]:
print(dataset[0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a medical AI assistant. Answer questions accurately and concisely.<|eot_id|><|start_header_id|>user<|end_header_id|>

What type of injury to the arm/elbow most often leads to supracondylar fractures?<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Supracondylar fractures most often occur after hyperextension injuries of the arm/elbow.<|eot_id|>


In [11]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # 30% less VRAM
)

model.print_trainable_parameters()

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.05.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.
Unsloth 2026.4.1 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 13,631,488 || all params: 8,043,892,736 || trainable%: 0.1695


In [12]:
lora_trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./lora_output",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        max_grad_norm=0.3,
        warmup_steps=5,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=True,
        report_to="none",
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/1000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [13]:
lora_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 514 | Num Epochs = 1 | Total steps = 33
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 13,631,488 of 8,043,892,736 (0.17% trained)


Step,Training Loss
10,1.842906
20,0.990254
30,0.908652


TrainOutput(global_step=33, training_loss=1.2175936337673303, metrics={'train_runtime': 427.138, 'train_samples_per_second': 1.203, 'train_steps_per_second': 0.077, 'total_flos': 5592587776942080.0, 'train_loss': 1.2175936337673303})

In [14]:
FastLanguageModel.for_inference(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
       

In [15]:
lora_outputs = []

for prompt in test_prompts:
    response = generate_response(model, tokenizer, prompt)
    lora_outputs.append((prompt, response))

In [16]:
for i, (prompt, response) in enumerate(lora_outputs, 1):
    print(f"Prompt {i}: {prompt}")
    print(f"LoRA Response: {response}")
    print("-" * 80)

Prompt 1: What are the possible causes of low PTH and high calcium levels?
LoRA Response: Low PTH and high calcium levels can be caused by familial hypocalciuric hypercalcemia, which is a benign condition that is characterized by low parathyroid hormone levels and high calcium levels. It is often associated with an autosomal dominant inheritance pattern, meaning that an affected parent has a 50% chance of passing the mutated gene to each child. Other possible causes of low PTH and high calcium levels include vitamin D deficiency, vitamin D toxicity, and primary hyperparathyroidism. Primary hyperparathyroidism is a condition in which one or more of the parathyroid glands are overactive and produce excess parathyroid hormone, leading to high calcium levels.
--------------------------------------------------------------------------------
Prompt 2: A patient presents with low ejection fraction. What type of cardiac dysfunction is this most commonly associated with?
LoRA Response: This pati

## LoRA Responses
Prompt 1: What are the possible causes of low PTH and high calcium levels?
LoRA Response: Low PTH and high calcium levels can be caused by familial hypocalciuric hypercalcemia, which is a benign condition that is characterized by low parathyroid hormone levels and high calcium levels. It is often associated with an autosomal dominant inheritance pattern, meaning that an affected parent has a 50% chance of passing the mutated gene to each child. Other possible causes of low PTH and high calcium levels include vitamin D deficiency, vitamin D toxicity, and primary hyperparathyroidism. Primary hyperparathyroidism is a condition in which one or more of the parathyroid glands are overactive and produce excess parathyroid hormone, leading to high calcium levels.
--------------------------------------------------------------------------------
Prompt 2: A patient presents with low ejection fraction. What type of cardiac dysfunction is this most commonly associated with?

LoRA Response: This patient is likely experiencing systolic heart failure. Systolic heart failure occurs when the heart muscle is weakened, making it less able to pump blood effectively. This can be caused by various factors, such as coronary artery disease, high blood pressure, or cardiomyopathy. Low ejection fraction, which is a measure of the heart's ability to pump blood, is a common indicator of systolic heart failure. Treatment for systolic heart failure may include medications to help the heart pump more efficiently, lifestyle changes, and in some cases, surgery or other interventions.
--------------------------------------------------------------------------------
Prompt 3: What does the combination of very high hematocrit and low EPO suggest?

LoRA Response: The combination of very high hematocrit and low EPO suggests that the patient has secondary polycythemia. Secondary polycythemia is a condition in which the body produces too many red blood cells in response to a secondary stimulus, such as a chronic disease or a condition that leads to hypoxia. In this case, the high hematocrit level indicates that there are too many red blood cells in the blood, and the low EPO level suggests that the body is not producing enough erythropoietin, a hormone that regulates red blood cell production. This combination of findings can help to distinguish secondary polycythemia from primary polycythemia, which is a condition in which the body produces too many red blood cells due
--------------------------------------------------------------------------------
Prompt 4: What condition is suggested by low cortisol, low sodium, and high potassium in an infant?

LoRA Response: Low cortisol, low sodium, and high potassium in an infant suggest adrenal insufficiency.
--------------------------------------------------------------------------------

In [17]:
base_model = model.unload()

dora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    bias="none",
    task_type="CAUSAL_LM",
    use_dora=True,   # only real difference from standard LoRA
)

base_model.enable_input_require_grads()
dora_model = get_peft_model(base_model, dora_config)
dora_model.print_trainable_parameters()

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:285: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


trainable params: 13,959,168 || all params: 8,044,220,416 || trainable%: 0.1735


In [18]:
dora_trainer = SFTTrainer(
    model=dora_model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=SFTConfig(
        output_dir="./dora_output",
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        learning_rate=2e-4,
        logging_steps=10,
        save_strategy="no",
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        optim="adamw_8bit",
        max_grad_norm=0.3,
        warmup_steps=5,
        dataset_text_field="text",
        max_seq_length=max_seq_length,
        packing=True,
        report_to="none",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    ),
)

Unsloth: Sample packing skipped (custom data collator detected).


In [19]:
dora_trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,000 | Num Epochs = 1 | Total steps = 63
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 13,959,168 of 8,044,220,416 (0.17% trained)


Step,Training Loss
10,1.867125
20,0.975859
30,0.932869
40,0.844001
50,0.821598
60,0.790188


TrainOutput(global_step=63, training_loss=1.0281044415065221, metrics={'train_runtime': 1159.3277, 'train_samples_per_second': 0.863, 'train_steps_per_second': 0.054, 'total_flos': 7115901741711360.0, 'train_loss': 1.0281044415065221})

In [20]:
dora_model.eval()
FastLanguageModel.for_inference(dora_model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict(
                  (defau

In [21]:
dora_outputs = []

for prompt in test_prompts:
    response = generate_response(dora_model, tokenizer, prompt)
    dora_outputs.append((prompt, response))

In [22]:
for i, (prompt, response) in enumerate(dora_outputs, 1):
    print(f"Prompt {i}: {prompt}")
    print(f"DoRA Response: {response}")
    print("-" * 80)

Prompt 1: What are the possible causes of low PTH and high calcium levels?
DoRA Response: Low PTH and high calcium levels can be caused by several factors, including familial hypocalciuric hypercalcemia (FHH) and parathyroid carcinoma. FHH is a rare genetic disorder that affects the regulation of calcium levels in the body, leading to high calcium levels and low PTH. Parathyroid carcinoma, on the other hand, is a type of cancer that affects the parathyroid glands, which produce PTH. In both cases, the high calcium levels can lead to a range of symptoms, including kidney stones, bone pain, and cardiovascular disease. Treatment for low PTH and high calcium levels will depend on the underlying cause and may involve medication, surgery, or other interventions. It is important to work with a
--------------------------------------------------------------------------------
Prompt 2: A patient presents with low ejection fraction. What type of cardiac dysfunction is this most commonly associate

In [23]:
for i in range(len(test_prompts)):
    print(f"Prompt {i+1}: {test_prompts[i]}")
    print("\nBASELINE:")
    print(baseline_outputs[i][1])
    print("\nLORA:")
    print(lora_outputs[i][1])
    print("\nDORA:")
    print(dora_outputs[i][1])
    print("\n" + "=" * 100 + "\n")

Prompt 1: What are the possible causes of low PTH and high calcium levels?

BASELINE:
Low parathyroid hormone (PTH) and high calcium levels can be caused by several conditions. Some possible causes include:

1. **Hypercalcemia of malignancy**: Certain types of cancer, such as squamous cell lung cancer, renal cell carcinoma, and lymphoma, can produce substances that stimulate the release of calcium into the blood.
2. **Vitamin D intoxication**: Excessive intake of vitamin D or its analogs can lead to increased absorption of calcium from the gut, resulting in high calcium levels and suppressed PTH secretion.
3. **Exogenous calcium intake**: Consuming large amounts of calcium supplements or calcium-rich foods can cause hypercalcemia.
4. **Granulomatous diseases**: Conditions like sarcoidosis, tuberculosis

LORA:
Low PTH and high calcium levels can be caused by familial hypocalciuric hypercalcemia, which is a benign condition that is characterized by low parathyroid hormone levels and high

| Prompt | Baseline | LoRA | DoRA |
|------|----------|------|------|
| Low PTH + high calcium | Broad list (malignancy, vitamin D, etc.) | Mentions FHH but includes incorrect causes | Mentions FHH + parathyroid carcinoma |
| Low ejection fraction | Correct: systolic heart failure (detailed) | Correct but more wordy | Correct but very brief |
| High hematocrit + low EPO | Correct: polycythemia vera | Incorrect: says secondary polycythemia | Vague and less precise |
| Low cortisol + low sodium + high potassium | Correct: CAH (salt-wasting) | General: adrenal insufficiency | Correct: CAH (detailed) |

A. LoRA and DoRA likely had similar training losses, but DoRA can sometimes converge better because it separates how weights are updated. In my results, DoRA responses were sometimes more focused, but not always more accurate than LoRA.

If trained for more epochs or on the full dataset, both models would likely improve. DoRA might show clearer advantages with more data, but LoRA would also become more consistent over time.

B. 4 bit quantization allows a large model to run on limited hardware by reducing memory usage. Without it, the model would require much more VRAM and would not run easily in Colab.

The trade off is slightly lower precision, which can affect performance. However, this loss is small compared to the benefit of being able to train the model efficiently, making it a reasonable trade off.

C. A major concern is that the model was trained on only 1,000 examples for one epoch, which is not enough for reliable real world performance. Some outputs were also incorrect, showing that the model is not fully dependable.

Additionally, the dataset is based on flashcards, which are simpler than real clinical situations. For real use, the model would need more training, testing, and oversight to ensure accuracy and safety.